# Lesson 3.5 — Variables in ADQL Queries

## Learning Objectives
- Use dashboard variables inside ADQL queries in Dash Studio
- Understand the variable injection syntax in AppDynamics
- Use template variables inside ADQL queries in Grafana
- Write parameterized queries that respond to user filter selections

**Reference**: [Variables in ADQL Queries](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/variables-in-adql-queries) | [Grafana ADQL with Template Variables](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-examples)

## 1. Why Variables Matter

A static ADQL query like:
```sql
SELECT count(*) FROM transactions WHERE transactionName = '/checkout'
```

...always shows data for `/checkout`. But what if your dashboard user wants to switch between BTs?

**Dashboard variables** let users select values interactively — a dropdown, multi-select, or text input — and those selections are **injected into ADQL queries**.

This turns a static dashboard into a dynamic investigation tool.

## 2. Variable Syntax in AppDynamics Dash Studio

In **Dash Studio**, variables use the `${variableName}` syntax:

```sql
-- Single-value variable
SELECT count(*) FROM transactions WHERE transactionName = '${selectedBT}'

-- Multi-select variable (IN clause)
SELECT transactionName, count(*) FROM transactions
WHERE transactionName IN (${selectedBTs})

-- Multiple variables combined
SELECT count(*), avg(responseTime) FROM transactions
WHERE transactionName = '${selectedBT}' AND country = '${selectedCountry}'
```

**Reference**: [Variables in ADQL Queries](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/dashboards-and-reports/dash-studio/data-binding/variables-in-adql-queries)

### Creating a Dashboard Variable in Dash Studio:
1. Open your dashboard in **Dash Studio**
2. Click the **Variables** icon (filter icon) in the top toolbar
3. Click **Add Variable**
4. Fill in:
   - **Name**: `selectedBT` (this is what you reference as `${selectedBT}`)
   - **Type**: Dropdown, Multi-Select, or Text Input
   - **Default value**: e.g., `/checkout`
   - **Options source**: Static list, or dynamic from a metric/ADQL query
5. Save, and the variable appears as a filter at the top of the dashboard

## 3. Variable Syntax in Grafana

In the **Grafana AppDynamics datasource**, template variables use Grafana's standard `${varName}` syntax, with type formatting modifiers:

```sql
-- Single variable (string value automatically quoted)
SELECT count(*) FROM transactions WHERE transactionName = '${transactionName}'

-- Multi-value variable with doublequote formatter (for IN clause)
SELECT distinct(transactionName), count(*) FROM transactions
WHERE transactionName IN (${transactionName:doublequote})

-- Multi-value variable with singlequote formatter
SELECT * FROM transactions
WHERE country IN (${country:singlequote})
LIMIT 50
```

**Reference**: [Grafana Analytics Query Examples](https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/#analytics-query-examples)

### Grafana variable formatting options:

| Format | Result for `['/checkout', '/search']` | Use case |
|--------|---------------------------------------|----------|
| `${var:doublequote}` | `"/checkout","/search"` | ADQL `IN` clause |
| `${var:singlequote}` | `'/checkout','/search'` | SQL `IN` clause |
| `${var:csv}` | `/checkout,/search` | comma-separated |
| `${var}` | `/checkout` (first value only) | Single value |

In [ ]:
# Demonstrate variable injection in ADQL queries

def inject_variable(query: str, variable_name: str, value, fmt: str = None) -> str:
    """
    Simulate how AppDynamics/Grafana injects dashboard variables into ADQL queries.
    
    fmt options: 'doublequote', 'singlequote', 'csv', None (raw)
    value: can be a string (single) or list (multi-select)
    """
    if isinstance(value, list):
        if fmt == 'doublequote':
            rendered = ','.join(f'"{v}"' for v in value)
        elif fmt == 'singlequote':
            rendered = ','.join(f"'{v}'" for v in value)
        elif fmt == 'csv':
            rendered = ','.join(str(v) for v in value)
        else:
            rendered = str(value[0])  # first value only
    else:
        rendered = str(value)
    
    # Replace both ${var:fmt} and ${var} patterns
    import re
    query = re.sub(rf'\${{\s*{re.escape(variable_name)}(?::[a-z]+)?\s*}}', rendered, query)
    return query

# Example 1: Dash Studio variable injection
dash_studio_query = """
SELECT series(eventTimestamp, '1m'), count(*) AS calls, avg(responseTime) AS avgRT
FROM transactions
WHERE transactionName = '${selectedBT}'
  AND userExperience IN (${selectedExperience})
"""

# User selects /checkout from dropdown, SLOW + ERROR from multi-select
result1 = inject_variable(dash_studio_query, "selectedBT", "/checkout")
result1 = inject_variable(result1, "selectedExperience", ["'SLOW'", "'VERY_SLOW'", "'ERROR'"])

print("=== Dash Studio Variable Injection ===")
print("Template:")
print(dash_studio_query.strip())
print("\nAfter injection (user selected: BT=/checkout, Experience=SLOW,VERY_SLOW,ERROR):")
print(result1.strip())

In [ ]:
# Example 2: Grafana template variable injection with :doublequote formatter

grafana_query = """
SELECT distinct(transactionName), count(*) AS calls, avg(responseTime) AS avgRT
FROM transactions
WHERE transactionName IN (${transactionName:doublequote})
"""

# Multi-select: user picked /checkout and /cart/add
selected_bts = ["/checkout", "/cart/add", "/search"]

result2 = inject_variable(grafana_query, "transactionName", selected_bts, fmt='doublequote')

print("=== Grafana :doublequote Variable Injection ===")
print("Template:")
print(grafana_query.strip())
print(f"\nUser selected: {selected_bts}")
print("\nAfter injection:")
print(result2.strip())
print()
print("Reference: https://grafana.com/docs/plugins/dlopes7-appdynamics-datasource/latest/query-editor/")

## 4. Dynamic Variable Sources

Instead of defining variable options as a static list, you can populate them dynamically.

### In Grafana — using AppDynamics query types as variable sources:

| Variable Type | Source Query | Example |
|---------------|-------------|----------|
| **Applications** | Query type: `Applications` | Lists all AppDynamics applications |
| **Tiers** | Query type: `Tiers` for an application | Lists all tiers in a specific app |
| **Metric Names** | Query type: `Metric Names` | Browse the metric hierarchy |

**Example Grafana workflow**:
1. Create a dashboard variable `application` → source: `Applications` query type
2. Create a variable `tier` → source: `Tiers` query, using `${application}` as the app
3. Use `${application}` and `${tier}` in your Metrics query type
4. Now users can pick any application AND tier — all panels update dynamically

### In Dash Studio — variable sources:
- **Static list**: comma-separated values you define
- **Metric-based**: populate options from a specific metric's values
- **ADQL-based**: run an ADQL query like `SELECT distinct(transactionName) FROM transactions` to populate the dropdown

In [ ]:
# Example 3: ADQL query that populates a Grafana variable dropdown
# This query returns all unique transaction names, suitable for a variable dropdown.

variable_population_query = """
SELECT distinct(transactionName)
FROM transactions
ORDER BY transactionName ASC
"""

print("Variable population query (for Grafana/Dash Studio dropdown):")
print(variable_population_query.strip())
print()
print("This would return:")

# Simulate the result from our dataset
import random
from datetime import datetime, timedelta

random.seed(42)
dataset = []
tx_names = ["/checkout", "/search", "/cart/add", "/cart/view", "/login", "/product/{id}"]
for tx in tx_names:
    print(f"  {{\"transactionName\": \"{tx}\"}}")

print()
print("Grafana would present these as selectable options in the variable dropdown.")

In [ ]:
# EXERCISE 3.5: Write parameterized ADQL queries
#
# For a Dash Studio operations dashboard with these variables:
#   ${app}         = selected application name (single-select)
#   ${tier}        = selected tier (single-select) 
#   ${experience}  = user experience filter (multi-select: SLOW, VERY_SLOW, ERROR)
#
# TASK: Write ADQL queries for each dashboard widget.

widgets = [
    {
        "widget": "Time Series: Transaction rate by user experience",
        "description": "Show count per minute, filtered by app and tier, grouped by userExperience",
        "adql": """-- TODO: Complete this query
SELECT series(eventTimestamp, '1m'), userExperience, count(*)
FROM transactions
WHERE applicationName = '${app}'
  AND ??? = '${tier}'"""
    },
    {
        "widget": "Table: Slowest transactions matching filter",
        "description": "List transactions where userExperience is in the selected set",
        "adql": """-- TODO: Complete this query
SELECT transactionName, responseTime, userExperience
FROM transactions
WHERE applicationName = '${app}'
  AND userExperience IN (${experience})
ORDER BY ???
LIMIT 20"""
    },
    {
        "widget": "Stat: Total error count",
        "description": "Count all errors for the selected app",
        "adql": """-- TODO: Complete this query
SELECT ???
FROM transactions
WHERE ??? AND errorOccurred = true"""
    },
]

for w in widgets:
    print(f"Widget: {w['widget']}")
    print(f"  Goal: {w['description']}")
    print(f"  Query template:")
    for line in w['adql'].split('\n'):
        print(f"    {line}")
    print()

## Summary

- **Dash Studio variables** use `${variableName}` syntax in ADQL queries
- **Grafana template variables** use `${varName}` with optional format modifiers (`:doublequote`, `:singlequote`, `:csv`)
- For multi-select IN clauses in Grafana, always use `${var:doublequote}`
- Variable options can be populated **statically** (fixed list) or **dynamically** (from ADQL distinct queries or AppDynamics query types)
- Parameterized queries are the foundation of useful, reusable dashboards

---

**Phase 3 Complete!** Proceed to [Phase 4 — Dash Studio](../phase-4-dash-studio/lesson-4.1-dashboards-overview.ipynb)